# MOOCCubeX raw JSON/TXT → Parquet trên Kaggle

Notebook này chuyển toàn bộ các file raw MOOCCubeX từ JSON Lines và TSV/TXT sang Parquet nén ZSTD.

Đặc điểm:

- tự tìm dataset root dưới /kaggle/input qua course.json;
- hỗ trợ cả layout gốc entities/relations/prerequisites và layout phẳng chứa mọi file trong cùng một root;
- chuẩn hóa tên logic về layout chính thức để output luôn ổn định;
- xử lý out-of-core bằng DuckDB, phù hợp với user-problem.json khoảng 21 GB;
- dùng schema tường minh cho các bảng lớn và nested;
- giữ cấu trúc nested gốc của course, user-video, video và user;
- xuất nhiều part Parquet có row group phù hợp để đọc song song;
- checkpoint theo từng bảng trong manifest.json;
- không im lặng bỏ qua JSON lỗi;
- chỉ ghi dữ liệu vào /kaggle/working/moocubex_parquet.

Đầu ra mỗi bảng nằm tại tables/<nhóm>/<tên_bảng>/*.parquet, kể cả khi đầu vào là root phẳng. Notebook này chỉ làm tầng raw-to-Parquet; việc bung segment, resource và tạo big-chapter LO nên nằm ở notebook tiếp theo.

In [ ]:
from __future__ import annotations

import csv
import json
import math
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

try:
    import duckdb
except ModuleNotFoundError:
    print("DuckDB chưa có trong image; đang cài từ package index...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb>=1.3"])
    import duckdb

print("Python:", sys.version.split()[0])
print("DuckDB:", duckdb.__version__)

# =========================
# CẤU HÌNH NGƯỜI DÙNG
# =========================

# Để None để tự tìm dưới /kaggle/input. Nếu có nhiều bản MOOCCubeX,
# đặt đường dẫn cụ thể, ví dụ "/kaggle/input/moocubex/mooccube2".
INPUT_ROOT: Optional[str] = None

OUTPUT_ROOT = Path("/kaggle/working/moocubex_parquet")

# None = chuyển tất cả JSON/TXT/TSV tìm thấy.
# Có thể chạy thử, ví dụ:
# ONLY_TABLES = ["entities/course.json", "relations/exercise-problem.txt"]
ONLY_TABLES: Optional[List[str]] = None

FORCE_REBUILD = False
STOP_ON_ERROR = True
IGNORE_JSON_ERRORS = False

THREADS = min(os.cpu_count() or 4, 64)
MEMORY_FRACTION = 0.75
ROW_GROUP_SIZE = 250_000
OUTPUT_FILE_SIZE = "512MB"
ZSTD_LEVEL = 3
JSON_SAMPLE_SIZE = 100_000
MAXIMUM_JSON_OBJECT_SIZE = 268_435_456  # 256 MiB; hữu ích cho user/video rất lớn

SUPPORTED_SUFFIXES = {".json", ".txt", ".tsv"}
OFFICIAL_TOP_DIRS = {"entities", "relations", "prerequisites"}

print("THREADS =", THREADS)
print("OUTPUT_ROOT =", OUTPUT_ROOT)

In [ ]:
EXPECTED_FILES = [
    "entities/reply.json",
    "entities/video.json",
    "entities/comment.json",
    "entities/course.json",
    "entities/other.json",
    "entities/paper.json",
    "entities/problem.json",
    "entities/school.json",
    "entities/teacher.json",
    "entities/user.json",
    "entities/concept.json",
    "relations/course-school.txt",
    "relations/course-teacher.txt",
    "relations/user-comment.txt",
    "relations/video_id-ccid.txt",
    "relations/comment-reply.txt",
    "relations/concept-other.txt",
    "relations/course-comment.txt",
    "relations/concept-video.txt",
    "relations/exercise-problem.txt",
    "relations/user-reply.txt",
    "relations/concept-comment.txt",
    "relations/concept-paper.txt",
    "relations/concept-problem.txt",
    "relations/concept-reply.json",
    "relations/course-field.json",
    "relations/reply-reply.txt",
    "relations/user-problem.json",
    "relations/user-video.json",
    "relations/user-xiaomu.json",
    "prerequisites/psy.json",
    "prerequisites/cs.json",
    "prerequisites/math.json",
]


# Ánh xạ basename về đường dẫn logic chính thức. Ngoài tên có dấu gạch ngang,
# chấp nhận cả alias dùng gạch dưới thường gặp khi người dùng tự đóng gói dataset.
CANONICAL_BY_BASENAME: Dict[str, str] = {}
for canonical_path in EXPECTED_FILES:
    basename = Path(canonical_path).name.lower()
    CANONICAL_BY_BASENAME[basename] = canonical_path
    CANONICAL_BY_BASENAME[basename.replace("-", "_")] = canonical_path


def discover_input_root(configured: Optional[str]) -> Path:
    if configured:
        root = Path(configured)
        if not root.exists():
            raise FileNotFoundError(f"INPUT_ROOT không tồn tại: {root}")
        return root

    kaggle_input = Path("/kaggle/input")
    if not kaggle_input.exists():
        raise FileNotFoundError("Không tìm thấy /kaggle/input. Notebook này được thiết kế cho Kaggle.")

    canonical = sorted(kaggle_input.rglob("entities/course.json"))
    if canonical:
        roots = [p.parent.parent for p in canonical]
        print("Các MOOCCubeX root dạng thư mục tìm thấy:")
        for candidate_root in roots:
            print(" -", candidate_root)
        return roots[0]

    # Hỗ trợ Kaggle Dataset đã gom course.json và các file còn lại vào cùng một root.
    loose = sorted(kaggle_input.rglob("course.json"))
    if loose:
        roots = [
            p.parent.parent if p.parent.name == "entities" else p.parent
            for p in loose
        ]
        print("Không thấy entities/course.json; dùng root chứa course.json:", roots[0])
        return roots[0]

    archives = sorted(
        p for p in kaggle_input.rglob("*")
        if p.is_file() and (
            p.suffix.lower() in {".zip", ".tar", ".tgz"} or
            p.name.lower().endswith(".tar.gz")
        )
    )
    archive_note = "\n".join(f" - {p}" for p in archives[:20])
    raise FileNotFoundError(
        "Không tìm thấy course.json đã giải nén dưới /kaggle/input. "
        "Nếu Kaggle Dataset chứa một archive, hãy giải nén nó vào /kaggle/temp "
        "và đặt INPUT_ROOT tới thư mục chứa course.json. Archive tìm thấy:\n"
        + archive_note
    )


INPUT_ROOT_PATH = discover_input_root(INPUT_ROOT).resolve()
print("INPUT_ROOT_PATH =", INPUT_ROOT_PATH)


def relative_key(path: Path) -> str:
    """Đường dẫn vật lý tương đối so với input root."""
    return path.resolve().relative_to(INPUT_ROOT_PATH).as_posix()


def logical_key(path: Path) -> str:
    """Khóa bảng ổn định, không phụ thuộc input là flat hay nested."""
    raw_rel = relative_key(path)
    if raw_rel in EXPECTED_FILES:
        return raw_rel
    return CANONICAL_BY_BASENAME.get(path.name.lower(), raw_rel)


all_candidates = sorted(
    p for p in INPUT_ROOT_PATH.rglob("*")
    if p.is_file() and p.suffix.lower() in SUPPORTED_SUFFIXES
)

official_candidates = [
    p for p in all_candidates
    if relative_key(p).split("/", 1)[0] in OFFICIAL_TOP_DIRS
]
known_flat_candidates = [
    p for p in all_candidates
    if logical_key(p) in EXPECTED_FILES
]

if official_candidates:
    INPUT_LAYOUT = "official_nested"
    SOURCE_FILES = official_candidates
elif known_flat_candidates:
    INPUT_LAYOUT = "flat_or_custom"
    SOURCE_FILES = known_flat_candidates
else:
    INPUT_LAYOUT = "generic"
    SOURCE_FILES = all_candidates

if ONLY_TABLES:
    requested = set(ONLY_TABLES)
    requested_basenames = {Path(x).name.lower() for x in requested}
    SOURCE_FILES = [
        p for p in SOURCE_FILES
        if (
            logical_key(p) in requested
            or relative_key(p) in requested
            or p.name.lower() in requested_basenames
        )
    ]

if not SOURCE_FILES:
    raise RuntimeError("Không có file JSON/TXT/TSV nào phù hợp để chuyển.")

# Không cho hai file vật lý ghi đè cùng một bảng logic.
logical_sources: Dict[str, List[Path]] = {}
for source_path in SOURCE_FILES:
    logical_sources.setdefault(logical_key(source_path), []).append(source_path)

logical_duplicates = {
    key: paths for key, paths in logical_sources.items() if len(paths) > 1
}
if logical_duplicates:
    details = "\n".join(
        f" - {key}: " + ", ".join(str(p) for p in paths)
        for key, paths in logical_duplicates.items()
    )
    raise RuntimeError(
        "Có nhiều file nguồn ánh xạ vào cùng một bảng logic; "
        "hãy đặt INPUT_ROOT cụ thể hơn hoặc loại file trùng:\n" + details
    )

existing = {logical_key(p) for p in SOURCE_FILES}
missing_expected = [x for x in EXPECTED_FILES if x not in existing]

total_source_bytes = sum(p.stat().st_size for p in SOURCE_FILES)
print("INPUT_LAYOUT =", INPUT_LAYOUT)
print(f"Số file sẽ chuyển: {len(SOURCE_FILES)}")
print(f"Tổng dung lượng raw: {total_source_bytes / 1024**3:.2f} GiB")
if missing_expected:
    print(f"Thiếu {len(missing_expected)} file trong danh sách phát hành chính thức.")
    for x in missing_expected:
        print(" -", x)

print("\n10 file lớn nhất:")
for p in sorted(SOURCE_FILES, key=lambda x: x.stat().st_size, reverse=True)[:10]:
    raw_rel = relative_key(p)
    logical_rel = logical_key(p)
    suffix = "" if raw_rel == logical_rel else f"  [source: {raw_rel}]"
    print(f"{p.stat().st_size / 1024**3:8.2f} GiB  {logical_rel}{suffix}")

In [ ]:
# Schema tường minh cho các bảng lớn/quan trọng.
# ID được lưu VARCHAR để giữ ổn định giữa những file dùng số và file dùng tiền tố.

JSON_SCHEMAS: Dict[str, Dict[str, str]] = {
    "entities/course.json": {
        "id": "VARCHAR",
        "name": "VARCHAR",
        "field": "VARCHAR[]",
        "prerequisites": "VARCHAR",
        "about": "VARCHAR",
        "resource": (
            "STRUCT("
            "titles VARCHAR[], "
            "resource_id VARCHAR, "
            "chapter VARCHAR, "
            "resource_type VARCHAR"
            ")[]"
        ),
    },
    "entities/problem.json": {
        "problem_id": "VARCHAR",
        "title": "VARCHAR",
        "content": "VARCHAR",
        "option": "JSON",
        "answer": "VARCHAR",
        "score": "DOUBLE",
        "type": "INTEGER",
        "typetext": "VARCHAR",
        "location": "VARCHAR",
        "context_id": "JSON",
        "exercise_id": "VARCHAR",
        "language": "VARCHAR",
    },
    "entities/video.json": {
        "ccid": "VARCHAR",
        "name": "VARCHAR",
        "start": "DOUBLE[]",
        "end": "DOUBLE[]",
        "text": "VARCHAR[]",
    },
    "entities/comment.json": {
        "id": "VARCHAR",
        "user_id": "VARCHAR",
        "text": "VARCHAR",
        "resource_id": "VARCHAR",
        "create_time": "VARCHAR",
    },
    "entities/reply.json": {
        "id": "VARCHAR",
        "user_id": "VARCHAR",
        "text": "VARCHAR",
        "create_time": "VARCHAR",
    },
    "entities/user.json": {
        "id": "VARCHAR",
        "name": "VARCHAR",
        "gender": "INTEGER",
        "school": "VARCHAR",
        "year_of_birth": "BIGINT",
        "course_order": "BIGINT[]",
        "enroll_time": "VARCHAR[]",
    },
    "relations/user-problem.json": {
        "log_id": "VARCHAR",
        "problem_id": "VARCHAR",
        "user_id": "VARCHAR",
        "is_correct": "INTEGER",
        "attempts": "BIGINT",
        "score": "DOUBLE",
        "submit_time": "VARCHAR",
    },
    "relations/user-video.json": {
        "user_id": "VARCHAR",
        "seq": (
            "STRUCT("
            "video_id VARCHAR, "
            "segment STRUCT("
            "start_point DOUBLE, "
            "end_point DOUBLE, "
            "speed DOUBLE, "
            "local_start_time BIGINT"
            ")[]"
            ")[]"
        ),
    },
}

TSV_COLUMNS: Dict[str, List[str]] = {
    "relations/course-school.txt": ["course_id", "school_id"],
    "relations/course-teacher.txt": ["course_id", "teacher_id"],
    "relations/user-comment.txt": ["user_id", "comment_id"],
    "relations/video_id-ccid.txt": ["video_id", "ccid"],
    "relations/comment-reply.txt": ["comment_id", "reply_id"],
    "relations/concept-other.txt": ["concept_id", "other_id"],
    "relations/course-comment.txt": ["course_id", "comment_id"],
    "relations/concept-video.txt": ["concept_id", "ccid"],
    "relations/exercise-problem.txt": ["exercise_id", "problem_id"],
    "relations/user-reply.txt": ["user_id", "reply_id"],
    "relations/concept-comment.txt": ["concept_id", "comment_id"],
    "relations/concept-paper.txt": ["concept_id", "paper_id"],
    "relations/concept-problem.txt": ["concept_id", "problem_id"],
    "relations/reply-reply.txt": ["parent_reply_id", "child_reply_id"],
}


def unique_basename_lookup(mapping: Dict[str, object]) -> Dict[str, object]:
    grouped: Dict[str, List[object]] = {}
    for key, value in mapping.items():
        grouped.setdefault(Path(key).name, []).append(value)
    return {name: values[0] for name, values in grouped.items() if len(values) == 1}


JSON_SCHEMA_BY_BASENAME = unique_basename_lookup(JSON_SCHEMAS)
TSV_COLUMNS_BY_BASENAME = unique_basename_lookup(TSV_COLUMNS)


def lookup_json_schema(rel: str, path: Path) -> Optional[Dict[str, str]]:
    return JSON_SCHEMAS.get(rel) or JSON_SCHEMA_BY_BASENAME.get(path.name)


def lookup_tsv_columns(rel: str, path: Path) -> Optional[List[str]]:
    return TSV_COLUMNS.get(rel) or TSV_COLUMNS_BY_BASENAME.get(path.name)


print("Explicit JSON schemas:", len(JSON_SCHEMAS))
print("Named TSV relations:", len(TSV_COLUMNS))

In [ ]:
def total_memory_bytes() -> int:
    try:
        return os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")
    except (ValueError, OSError, AttributeError):
        return 30 * 1024**3


total_mem = total_memory_bytes()
memory_limit_gib = max(4, int(total_mem * MEMORY_FRACTION / 1024**3))

scratch_base = Path("/kaggle/temp")
if not scratch_base.exists():
    scratch_base = OUTPUT_ROOT / "_scratch"
DUCKDB_TEMP = scratch_base / "moocubex_duckdb_tmp"
DUCKDB_TEMP.mkdir(parents=True, exist_ok=True)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

con = duckdb.connect(database=":memory:")
con.execute(f"SET threads={THREADS}")
con.execute(f"SET memory_limit='{memory_limit_gib}GB'")
con.execute(f"SET temp_directory='{str(DUCKDB_TEMP).replace(chr(39), chr(39) * 2)}'")
con.execute("SET preserve_insertion_order=false")
con.execute("SET enable_progress_bar=true")

print(f"DuckDB memory_limit = {memory_limit_gib} GiB")
print("DuckDB temp_directory =", DUCKDB_TEMP)
print("Dung lượng trống /kaggle/working:",
      f"{shutil.disk_usage('/kaggle/working').free / 1024**3:.2f} GiB")

In [ ]:
MANIFEST_PATH = OUTPUT_ROOT / "manifest.json"


def sql_string(value: str) -> str:
    return "'" + value.replace("'", "''") + "'"


def sql_identifier(value: str) -> str:
    return '"' + value.replace('"', '""') + '"'


def detect_json_format(path: Path) -> str:
    with path.open("rb") as f:
        prefix = f.read(65_536)
    prefix = prefix.lstrip()
    if prefix.startswith(b"["):
        return "array"
    return "newline_delimited"


def infer_tsv_column_count(path: Path, limit: int = 1000) -> int:
    counts: Dict[int, int] = {}
    with path.open("r", encoding="utf-8", errors="replace") as f:
        for i, line in enumerate(f):
            if i >= limit:
                break
            line = line.rstrip("\r\n")
            if not line:
                continue
            n = line.count("\t") + 1
            counts[n] = counts.get(n, 0) + 1
    if not counts:
        raise ValueError(f"Không tìm thấy dòng dữ liệu trong {path}")
    return max(counts.items(), key=lambda x: x[1])[0]


def output_dir_for(rel: str) -> Path:
    rel_path = Path(rel)
    clean_parts = [part.replace("-", "_") for part in rel_path.with_suffix("").parts]
    out_dir = OUTPUT_ROOT / "tables"
    for part in clean_parts:
        out_dir = out_dir / part
    return out_dir


def build_json_query(path: Path, rel: str) -> Tuple[str, str, str]:
    json_format = detect_json_format(path)
    schema = lookup_json_schema(rel, path)
    base_options = [
        f"format={sql_string(json_format)}",
        f"maximum_object_size={MAXIMUM_JSON_OBJECT_SIZE}",
    ]

    if json_format == "newline_delimited":
        base_options.append(
            f"ignore_errors={'true' if IGNORE_JSON_ERRORS else 'false'}"
        )

    if schema:
        columns_literal = "{" + ", ".join(
            f"{sql_identifier(name)}: {sql_string(dtype)}" for name, dtype in schema.items()
        ) + "}"
        options = base_options + [
            "auto_detect=false",
            f"columns={columns_literal}",
        ]
        selected = ", ".join(sql_identifier(name) for name in schema)
        relation = (
            f"read_json({sql_string(str(path))}, "
            + ", ".join(options)
            + ")"
        )
        return f"SELECT {selected} FROM {relation}", json_format, "explicit"

    options = base_options + [
        f"sample_size={JSON_SAMPLE_SIZE}",
        "maximum_depth=-1",
    ]
    relation = (
        f"read_json_auto({sql_string(str(path))}, "
        + ", ".join(options)
        + ")"
    )
    return f"SELECT * FROM {relation}", json_format, "auto"


def build_tsv_query(path: Path, rel: str) -> Tuple[str, str, str]:
    columns = lookup_tsv_columns(rel, path)
    mode = "named"
    if not columns:
        n_cols = infer_tsv_column_count(path)
        columns = [f"col_{i + 1}" for i in range(n_cols)]
        mode = "generic"

    columns_literal = "{" + ", ".join(
        f"{sql_identifier(name)}: 'VARCHAR'" for name in columns
    ) + "}"

    # Đây là relation TSV thô, không phải RFC 4180 CSV. Dấu quote trong ID/text
    # phải được giữ như ký tự thường; nếu để auto-detect, quote lẻ có thể làm parser
    # rơi vào trạng thái không hợp lệ (đã quan sát ở concept-paper.txt).
    relation = (
        f"read_csv({sql_string(str(path))}, "
        "delim='\\t', header=false, auto_detect=false, "
        "quote='', escape='', strict_mode=false, "
        f"columns={columns_literal}, null_padding=true, parallel=true)"
    )
    selected = ", ".join(sql_identifier(name) for name in columns)
    return f"SELECT {selected} FROM {relation}", "tsv", mode


def build_source_query(path: Path, rel: str) -> Tuple[str, str, str]:
    if path.suffix.lower() == ".json":
        return build_json_query(path, rel)
    if path.suffix.lower() in {".txt", ".tsv"}:
        return build_tsv_query(path, rel)
    raise ValueError(f"Định dạng chưa hỗ trợ: {path}")


def load_manifest() -> Dict[str, dict]:
    if not MANIFEST_PATH.exists():
        return {}
    with MANIFEST_PATH.open("r", encoding="utf-8") as f:
        payload = json.load(f)
    return {row["relative_path"]: row for row in payload}


def save_manifest(manifest: Dict[str, dict]) -> None:
    records = sorted(manifest.values(), key=lambda x: x["relative_path"])
    tmp_path = MANIFEST_PATH.with_suffix(".json.tmp")
    with tmp_path.open("w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    tmp_path.replace(MANIFEST_PATH)


manifest = load_manifest()


def has_parquet_files(out_dir: Path) -> bool:
    return out_dir.exists() and any(out_dir.glob("*.parquet"))


def remove_scoped_output(out_dir: Path) -> None:
    output_resolved = OUTPUT_ROOT.resolve()
    target_resolved = out_dir.resolve()
    if output_resolved not in target_resolved.parents:
        raise RuntimeError(f"Từ chối xóa path ngoài OUTPUT_ROOT: {out_dir}")
    if out_dir.exists():
        shutil.rmtree(out_dir)


def convert_one(path: Path) -> dict:
    raw_rel = relative_key(path)
    rel = logical_key(path)
    out_dir = output_dir_for(rel)
    previous = manifest.get(rel)

    if (
        not FORCE_REBUILD
        and previous
        and previous.get("status") == "success"
        and has_parquet_files(out_dir)
    ):
        print("SKIP", rel)
        return previous

    source_query, source_format, schema_mode = build_source_query(path, rel)
    remove_scoped_output(out_dir)
    out_dir.parent.mkdir(parents=True, exist_ok=True)

    copy_sql = f"""
        COPY ({source_query})
        TO {sql_string(str(out_dir))}
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD,
            COMPRESSION_LEVEL {ZSTD_LEVEL},
            ROW_GROUP_SIZE {ROW_GROUP_SIZE},
            FILE_SIZE_BYTES {sql_string(OUTPUT_FILE_SIZE)},
            PER_THREAD_OUTPUT true
        )
    """

    started = time.time()
    record = {
        "relative_path": rel,
        "source_relative_path": raw_rel,
        "source_path": str(path),
        "source_bytes": path.stat().st_size,
        "source_format": source_format,
        "schema_mode": schema_mode,
        "output_dir": str(out_dir),
        "status": "running",
        "row_count": None,
        "output_files": 0,
        "output_bytes": 0,
        "elapsed_seconds": None,
        "error": None,
    }
    manifest[rel] = record
    save_manifest(manifest)

    print("\nCONVERT", rel)
    try:
        con.execute(copy_sql)
        parquet_files = sorted(out_dir.glob("*.parquet"))
        if not parquet_files:
            raise RuntimeError("DuckDB không tạo file Parquet nào.")

        parquet_glob = str(out_dir / "*.parquet")
        row_count = con.execute(
            f"SELECT count(*) FROM read_parquet({sql_string(parquet_glob)})"
        ).fetchone()[0]

        record.update({
            "status": "success",
            "row_count": int(row_count),
            "output_files": len(parquet_files),
            "output_bytes": sum(p.stat().st_size for p in parquet_files),
            "elapsed_seconds": round(time.time() - started, 3),
        })
        print(
            f"OK rows={record['row_count']:,} "
            f"files={record['output_files']} "
            f"size={record['output_bytes'] / 1024**2:.1f} MiB "
            f"time={record['elapsed_seconds']:.1f}s"
        )
    except Exception as exc:
        record.update({
            "status": "error",
            "elapsed_seconds": round(time.time() - started, 3),
            "error": f"{type(exc).__name__}: {exc}",
        })
        print("ERROR", rel, record["error"])
        if STOP_ON_ERROR:
            manifest[rel] = record
            save_manifest(manifest)
            raise

    manifest[rel] = record
    save_manifest(manifest)
    return record


print("Manifest hiện có:", len(manifest), "bảng")

In [ ]:
# Chạy chuyển đổi.
# Sắp file nhỏ trước để schema/permission lỗi xuất hiện sớm;
# hai fact table lớn sẽ chạy sau.

ordered_sources = sorted(SOURCE_FILES, key=lambda p: (p.stat().st_size, logical_key(p)))

run_started = time.time()
for index, source_path in enumerate(ordered_sources, start=1):
    raw_rel = relative_key(source_path)
    logical_rel = logical_key(source_path)
    source_note = "" if raw_rel == logical_rel else f" (source: {raw_rel})"
    print(f"\n[{index}/{len(ordered_sources)}] {logical_rel}{source_note}")
    convert_one(source_path)

print("\nHoàn thành sau", round((time.time() - run_started) / 60, 2), "phút")

In [ ]:
# Xuất manifest dạng Parquet và kiểm tra kết quả.

manifest = load_manifest()
records = sorted(manifest.values(), key=lambda x: x["relative_path"])

manifest_records_json = OUTPUT_ROOT / "manifest_records.json"
with manifest_records_json.open("w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False)

manifest_parquet = OUTPUT_ROOT / "manifest.parquet"
if manifest_parquet.exists():
    manifest_parquet.unlink()

con.execute(
    f"""
    COPY (
        SELECT *
        FROM read_json_auto(
            {sql_string(str(manifest_records_json))},
            format='array'
        )
    )
    TO {sql_string(str(manifest_parquet))}
    (FORMAT PARQUET, COMPRESSION ZSTD, COMPRESSION_LEVEL {ZSTD_LEVEL})
    """
)

successes = [r for r in records if r["status"] == "success"]
failures = [r for r in records if r["status"] != "success"]

print("Success:", len(successes))
print("Failure/incomplete:", len(failures))
print("Total rows:", f"{sum((r.get('row_count') or 0) for r in successes):,}")
print(
    "Total Parquet size:",
    f"{sum((r.get('output_bytes') or 0) for r in successes) / 1024**3:.2f} GiB"
)

if failures:
    print("\nCác bảng chưa hoàn thành:")
    for row in failures:
        print(" -", row["relative_path"], row.get("error"))

# Kiểm tra các bảng trọng yếu.
critical = [
    "entities/course.json",
    "entities/problem.json",
    "relations/user-problem.json",
    "relations/exercise-problem.txt",
    "relations/user-video.json",
    "entities/video.json",
    "entities/comment.json",
    "relations/course-comment.txt",
    "relations/user-comment.txt",
    "relations/video_id-ccid.txt",
]

print("\nKiểm tra bảng trọng yếu:")
for rel in critical:
    row = manifest.get(rel)
    if row and row.get("status") == "success":
        print(
            f" OK {rel:40s} "
            f"rows={row['row_count']:,} "
            f"size={row['output_bytes'] / 1024**2:.1f} MiB"
        )
    else:
        print(" --", rel, "không có hoặc chưa thành công")

catalog = {
    "input_root": str(INPUT_ROOT_PATH),
    "output_root": str(OUTPUT_ROOT),
    "created_with_duckdb": duckdb.__version__,
    "table_count": len(successes),
    "tables": {
        row["relative_path"]: {
            "parquet_glob": str(Path(row["output_dir"]) / "*.parquet"),
            "row_count": row["row_count"],
            "source_format": row["source_format"],
            "schema_mode": row["schema_mode"],
        }
        for row in successes
    },
}
with (OUTPUT_ROOT / "catalog.json").open("w", encoding="utf-8") as f:
    json.dump(catalog, f, ensure_ascii=False, indent=2)

(OUTPUT_ROOT / "_SUCCESS").write_text(
    "MOOCCubeX raw-to-Parquet conversion completed.\n",
    encoding="utf-8",
)

print("\nOutput:", OUTPUT_ROOT)
print("Manifest:", manifest_parquet)
print("Catalog:", OUTPUT_ROOT / "catalog.json")
print("Marker:", OUTPUT_ROOT / "_SUCCESS")

## Cách đọc dữ liệu sau chuyển đổi

Dù input đặt trong entities/relations/prerequisites hay gom tất cả file vào một root, notebook vẫn chuẩn hóa output về cùng layout logic.

Ví dụ với DuckDB:

    SELECT *
    FROM read_parquet(
      '/kaggle/working/moocubex_parquet/tables/relations/user_problem/*.parquet'
    )
    LIMIT 10;

Ví dụ với Polars:

    import polars as pl

    user_problem = pl.scan_parquet(
        '/kaggle/working/moocubex_parquet/tables/relations/user_problem/*.parquet'
    )

Sau khi Save & Run All thành công, thư mục moocubex_parquet trong Notebook Output có thể được tạo thành một Kaggle Dataset mới. Notebook tiếp theo nên đọc Parquet này để:

1. bung course.resource và tạo resource-to-big-chapter bridge;
2. bung user-video.seq.segment;
3. chuẩn hóa problem/exercise và user/problem;
4. tạo các fixed calendar phase không chồng lấn;
5. tính COELO, ACELO và LOQV2D theo phase.